## 1. Lade vortrainiertes Modell

In [5]:
import sys

# verhindere, dass torchvision geladen wird
sys.modules["torchvision"] = None

# importiere verbliebene notwendige Bibliotheken
import torch
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from transformers import AutoTokenizer, AutoModel

# waehle das zu importierende Modell
MODEL_ID = "google-bert/bert-base-uncased"
MAX_TOKENS = 32  # includes [CLS], [SEP], and possible subword tokens

# lade Modell
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID, attn_implementation="eager")
model.eval()

print(f"Model {MODEL_ID} wurde geladen: {model.config.num_hidden_layers} Layers, "
      f"{model.config.num_attention_heads} Attention Heads pro Layer")

Model google-bert/bert-base-uncased wurde geladen: 12 Layers, 12 Attention Heads pro Layer


## 2. Analysiere (Self-)Attention

Editiere den Satz, waehle eine Layer und einen Attention Head und klicke auf **Zeige Attention**.

In [4]:
# definiere minimalistisches User Interface mit vorgewaehltem Satz
sentence = widgets.Textarea(
    value="I swam across the river to get to the other bank.",
    description="Satz:",
    style={"description_width": "75px"},
    layout=widgets.Layout(width="98%", height="75px"),
)
layer = widgets.Dropdown(
    options=[(f"Layer {i + 1}", i) for i in range(model.config.num_hidden_layers)],
    description="Layer:",value=2,
)
head = widgets.Dropdown(
    options=[(f"Head {i + 1}", i) for i in range(model.config.num_attention_heads)],
    description="Head:",value=10,
)
button = widgets.Button(description="Zeige Attention", button_style="primary")
output = widgets.Output()

# Routine die beim Klick auf "Zeige Attention" ausgefuehrt wird
def show_attention(_=None):
    with output:
        clear_output(wait=True)
        text = sentence.value.strip()
        if not text:
            print("Bitte Satz eingeben.")
            return

        # erzeuge Tokens aus Satz
        encoded = tokenizer(text, return_tensors="pt", truncation=True,
                            max_length=MAX_TOKENS)
        tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])

        # lese Attention Gewichte aus
        with torch.inference_mode():
            attentions = model(**encoded, output_attentions=True).attentions
        weights = attentions[layer.value][0, head.value].cpu().numpy()

        # plotte Heat Map
        n = len(tokens)
        fig, ax = plt.subplots(figsize=(max(6, 0.43 * n), max(5, 0.40 * n)))
        image = ax.imshow(weights, cmap="Blues", vmin=0,
                          vmax=max(0.25, float(weights.max())))
        ax.set_xticks(range(n), tokens, rotation=90)
        ax.set_yticks(range(n), tokens)
        ax.set_xlabel("Key Token (Ziel)")
        ax.set_ylabel("Query Token (Ausgangspunkt)")
        ax.set_title(f"Self Attention Gewichte: Layer {layer.value + 1}, Head {head.value + 1}")
        fig.colorbar(image, ax=ax, label="Attention Wert", shrink=0.78)
        fig.tight_layout()
        #plt.savefig("output.png",dpi=300)
        plt.show()
        plt.close(fig)


# verbinde obige Routine mit dem Button
button.on_click(show_attention)
# zeige User Interface an
display(widgets.VBox([sentence, widgets.HBox([layer, head, button]), output]))
# rufe sofort einmal die obige Routine auf, um gleich am Anfang schon eine Heat Map zu haben
show_attention()